# Project 2: predicting repayment difficulty
Run each cell from top to bottom. `TARGET = 1` means repayment difficulty.

This is our working model before manual error analysis. It uses a stratified split, employment cleanup, missing-value flags, and a 64 → 32 → 1 neural network. We choose the best epoch using validation AUC, then choose a classification threshold using validation F1. The test set stays unused.

## 1. Import packages and choose settings

In [ ]:
from pathlib import Path
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

SEED = 42
EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

## 2. Load the data
On Colab, upload `application_train.csv` to `/content`. Locally, keep it in the project folder.

In [ ]:
data_path = Path("application_train.csv")
if not data_path.exists():
    data_path = Path("/content/application_train.csv")
if not data_path.exists():
    data_path = Path.home() / "Downloads" / "Project 2 DATA" / "application_train.csv"

df = pd.read_csv(data_path)
applicant_ids = df["SK_ID_CURR"].copy()
y = df["TARGET"].copy()
X = df.drop(columns=["TARGET", "SK_ID_CURR"]).copy()
print("Applicants:", len(df))

## 3. Fix the employment placeholder and split the rows
The value `365243` is treated as missing, with a separate flag to preserve that information. All splits use the same seed: 60% training, 20% validation, 20% test.

In [ ]:
employment_placeholder = X["DAYS_EMPLOYED"].eq(365243)
X["DAYS_EMPLOYED_PLACEHOLDER"] = employment_placeholder.astype(int)
X.loc[employment_placeholder, "DAYS_EMPLOYED"] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)
X_train, X_val = X_train.copy(), X_val.copy()
print("Train / validation / test:", len(X_train), len(X_val), len(X_test))

## 4. Choose how each column is processed
Categories are one-hot encoded. Numeric measurements are standardized. Binary flags stay as 0/1.

In [ ]:
# Categorical columns
categorical_cols = [
    'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
    'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE',
    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
]

In [ ]:
# Numeric measurements
scale_cols = [
    'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
    'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'CNT_FAM_MEMBERS',
    'HOUR_APPR_PROCESS_START', 'EXT_SOURCE_1', 'EXT_SOURCE_2',
    'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG',
    'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG',
    'ELEVATORS_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG',
    'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG',
    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE',
    'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE',
    'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI',
    'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI',
    'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI',
    'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_MEDI', 'TOTALAREA_MODE', 'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'DAYS_LAST_PHONE_CHANGE', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR',
]

In [ ]:
# Binary flags
binary_cols = [
    'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
    'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL',
    'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY',
    'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4',
    'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7',
    'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10',
    'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13',
    'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16',
    'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19',
    'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21', 'DAYS_EMPLOYED_PLACEHOLDER',
]

## 5. Add missing-value flags
A flag records whether a numeric value was missing before imputation. We select these columns from training data only. Employment already has an equivalent flag, so it is not duplicated.

In [ ]:
missing_columns = [
    column for column in scale_cols
    if column != "DAYS_EMPLOYED" and X_train[column].isna().any()
]

for column in missing_columns:
    flag = column + "_MISSING"
    X_train[flag] = X_train[column].isna().astype(int)
    X_val[flag] = X_val[column].isna().astype(int)

missing_flags = [column + "_MISSING" for column in missing_columns]
model_binary_cols = binary_cols + missing_flags
print("Added numeric missingness flags:", len(missing_flags))

## 6. Fit preprocessing on training data only
Validation uses the transformations learned from training. We do not transform or score test data yet.

In [ ]:
preprocessor = ColumnTransformer([
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),

    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), scale_cols),

    ("binary", SimpleImputer(strategy="most_frequent"), model_binary_cols)
], sparse_threshold=0)

train_array = preprocessor.fit_transform(X_train).astype(np.float32)
val_array = preprocessor.transform(X_val).astype(np.float32)
assert np.isfinite(train_array).all() and np.isfinite(val_array).all()
print("Training shape:", train_array.shape)

## 7. Create tensors and the neural network
The model outputs logits. BCEWithLogitsLoss handles them during training; sigmoid converts them to probabilities during evaluation.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

train_X_tensor = torch.from_numpy(train_array).to(device)
val_X_tensor = torch.from_numpy(val_array).to(device)
train_y_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32, device=device)
val_y_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32, device=device)

train_loader = DataLoader(
    TensorDataset(train_X_tensor, train_y_tensor),
    batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED)
)

model = nn.Sequential(
    nn.Linear(train_array.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## 8. Train and remember the best validation epoch
Training and validation metrics are computed at the same end-of-epoch weights. This makes the two curves comparable.

In [ ]:
best_auc = -np.inf
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_X).squeeze(1), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        train_logits = model(train_X_tensor).squeeze(1)
        val_logits = model(val_X_tensor).squeeze(1)
        train_prob = torch.sigmoid(train_logits).cpu().numpy()
        val_prob = torch.sigmoid(val_logits).cpu().numpy()
        train_loss = loss_fn(train_logits, train_y_tensor).item()
        val_loss = loss_fn(val_logits, val_y_tensor).item()

    train_auc = roc_auc_score(y_train, train_prob)
    val_auc = roc_auc_score(y_val, val_prob)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                    "train_auc": train_auc, "val_auc": val_auc})
    print(f"Epoch {epoch:02d}: train AUC={train_auc:.4f}, validation AUC={val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        best_epoch = epoch
        best_weights = copy.deepcopy(model.state_dict())

model.load_state_dict(best_weights)
model.eval()
with torch.no_grad():
    val_probabilities = torch.sigmoid(model(val_X_tensor).squeeze(1)).cpu().numpy()

print(f"Best epoch: {best_epoch}; validation AUC: {best_auc:.5f}")

## 9. Check for overfitting

In [ ]:
history = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

for ax, metric in zip(axes, ["auc", "loss"]):
    ax.plot(history["epoch"], history["train_" + metric], label="Training")
    ax.plot(history["epoch"], history["val_" + metric], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Best epoch")
    ax.set(xlabel="Epoch", ylabel=metric.upper())
    ax.legend()

plt.show()

## 10. Choose a threshold using validation F1
Search 1%–50% in 0.1-percentage-point steps. This changes predicted labels, not the model or AUC. Nearby thresholds may be equally useful; the exact maximum is provisional.

In [ ]:
threshold_results = []
for step in range(10, 501):
    threshold = step / 1000
    predictions = val_probabilities >= threshold
    threshold_results.append({
        "threshold": threshold,
        "f1": f1_score(y_val, predictions, zero_division=0),
        "precision": precision_score(y_val, predictions, zero_division=0),
        "recall": recall_score(y_val, predictions, zero_division=0),
        "accuracy": accuracy_score(y_val, predictions)
    })

threshold_results = pd.DataFrame(threshold_results)
best_row = threshold_results.loc[threshold_results["f1"].idxmax()]
best_threshold = float(best_row["threshold"])
print(f"F1-selected threshold: {best_threshold:.1%}")
threshold_results.loc[
    threshold_results["threshold"].isin([0.10, 0.125, best_threshold, 0.15, 0.50])
].round(4)

## 11. Keep predictions ready for error analysis
The original features remain in `df`; match applicants using `SK_ID_CURR`. This table is preparation, not completed manual error analysis.

In [ ]:
validation_results = pd.DataFrame({
    "SK_ID_CURR": applicant_ids.loc[X_val.index].to_numpy(),
    "actual_TARGET": y_val.to_numpy(),
    "predicted_probability": val_probabilities,
    "predicted_label": (val_probabilities >= best_threshold).astype(int)
})
validation_results.head()

## What we have learned and what remains
Missingness indicators improved mean validation AUC from 0.74521 to 0.74673 across three training seeds on one stratified split. The ratios did not improve consistently. Training longer led to overfitting. These findings support this working baseline, not a claim of statistical significance.

The full experiment history is preserved in `Project_2_experiments_archive.ipynb` and `feature_experiment_runs/` in the project folder. The previous, more detailed version is backed up in `diagnostics/before_readability.ipynb`.

For the assignment, we still need manual validation error analysis and its spreadsheet, fairness evaluation, and a justified comparison of random, stratified and nonrandom splits. The earlier loan-amount split represents a distribution shift; its deployment justification remains unresolved. Previously reported split scores came from different validation populations and should not be treated as tuning results.

The full validation set has already informed modeling and threshold selection. An earlier 80/20 test result was also viewed, overlapping the current test subset; disclose this in the report. Finalize the eventual model after error-analysis iterations and fairness checks, then perform final evaluation. When adding final test evaluation, create the same flags listed in `missing_columns` and use the already fitted preprocessor—do not refit it.